In [84]:
# importing necessary libraries
import pandas as pd
import numpy as np
import ast


In [85]:
# retrieve the data
df = pd.read_csv('../../data/raw/tmdb/tmdb_raw_2019_2026.csv')
df.head(5)

,tmdb_id,title,original_title,popularity,vote_average,vote_count,release_date,original_language,genre_ids,adult
0,299534,Avengers: Endgame,Avengers: Endgame,98.7212,8.200,28768,2019-04-24,en,"[12, 878, 28]",False
1,496243,Parasite,기생충,35.6564,8.490,21362,2019-05-30,ko,"[35, 53, 18]",False
2,466272,Once Upon a Time... in Hollywood,Once Upon a Time... in Hollywood,34.9897,7.419,15284,2019-07-24,en,"[35, 18, 53]",False
3,475557,Joker,Joker,34.4579,8.100,28367,2019-10-01,en,"[80, 53, 18]",False
4,429617,Spider-Man: Far From Home,Spider-Man: Far From Home,33.8307,7.392,17646,2019-06-28,en,"[28, 12, 878]",False


In [86]:
df.columns

Index(['tmdb_id', 'title', 'original_title', 'popularity', 'vote_average',
       'vote_count', 'release_date', 'original_language', 'genre_ids',
       'adult'],
      dtype='str')

In [87]:
df.dtypes

tmdb_id                int64
title                    str
original_title           str
popularity           float64
vote_average         float64
vote_count             int64
release_date             str
original_language        str
genre_ids                str
adult                   bool
dtype: object

## let's inspect each column and treat problems one by one

In [88]:
df.isnull().sum()

tmdb_id              0
title                1
original_title       1
popularity           0
vote_average         0
vote_count           0
release_date         0
original_language    0
genre_ids            0
adult                0
dtype: int64

In [89]:
df[df["title"].isnull()]

,tmdb_id,title,original_title,popularity,vote_average,vote_count,release_date,original_language,genre_ids,adult
79496,1697128,NaN,NaN,1.568,0.0,0,2026-05-14,en,"[18, 35]",False


In [90]:
df = df.dropna(subset=["title"])
print(f"Lignes apres suppression du titre manquant : {len(df)}")

Lignes apres suppression du titre manquant : 79999


### checking for duplicate movies (same tmdb_id appearing more than once)

In [91]:
df.duplicated(subset="tmdb_id").sum()

np.int64(732)

In [92]:

# 1. Supprimer les doublons exacts d'identifiant TMDB
print("Doublons TMDB avant :", df.duplicated(subset="tmdb_id").sum())

df = df.drop_duplicates(subset="tmdb_id", keep="first").copy()

print("Nombre de lignes après déduplication :", len(df))
print("Doublons TMDB restants :", df.duplicated(subset="tmdb_id").sum())


Doublons TMDB avant : 732
Nombre de lignes après déduplication : 79267
Doublons TMDB restants : 0


## release_date: converting from text to a real date, and extracting the year
for consistency with the other datasets (imdb, rotten tomatoes)

In [93]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df[["release_date", "release_year"]].head(10)

,release_date,release_year
0,2019-04-24,2019
1,2019-05-30,2019
2,2019-07-24,2019
3,2019-10-01,2019
4,2019-06-28,2019
5,2019-06-19,2019
6,2019-11-20,2019
7,2019-11-27,2019
8,2019-12-03,2019
9,2019-07-31,2019


In [94]:
# checking if any release_date failed to convert (would show up as NaT)
df["release_date"].isnull().sum()

np.int64(0)

la fonction de nettoyage des titres

In [95]:
import re, unicodedata

def normalize_title(title):
    title = unicodedata.normalize("NFKD", str(title).lower())
    title = "".join(c for c in title if not unicodedata.combining(c))
    title = re.sub(r"[^a-z\s0-9]", "", title)
    return re.sub(r"\s+", " ", title).strip()

mesurer avant de supprimer

In [96]:
df["title_norm"] = df["title"].map(normalize_title)
key = ["title_norm", "release_year"]

# les titres sans caractère latin (coréen, chinois...) deviennent vides : on ne les compare pas
valid = df["title_norm"] != ""
print("titres sans caractère latin (non comparés) :", (~valid).sum())
print("copies en trop (titre normalisé + année) :", (valid & df.duplicated(subset=key)).sum())

titres sans caractère latin (non comparés) : 251
copies en trop (titre normalisé + année) : 2553


In [97]:
same = df[df.duplicated(subset=["title", "release_year"], keep=False)].sort_values(["title", "release_date"])
print(len(same), "lignes partagent le même titre et la même date")
same.head(20)

3971 lignes partagent le même titre et la même date


,tmdb_id,title,original_title,popularity,vote_average,vote_count,release_date,original_language,genre_ids,adult,release_year,title_norm
74738,1609606,119 Emergency Call 2026 YOKOHAMA BLACKOUT,119エマージェンシーコール2026 YOKOHAMA BLACKOUT,2.1256,5.500,2,2026-01-03,ja,[18],False,2026,119 emergency call 2026 yokohama blackout
79293,1586473,119 Emergency Call 2026 YOKOHAMA BLACKOUT,119エマージェンシーコール2026 YOKOHAMA BLACKOUT,1.5850,0.000,0,2026-01-03,ja,[18],False,2026,119 emergency call 2026 yokohama blackout
9573,895852,7,7,1.3455,7.800,2,2019-01-01,en,[],False,2019,7
5802,647404,7,7,1.6557,6.400,7,2019-12-05,ru,[99],False,2019,7
51564,1201056,A Family,Une famille,2.5506,7.094,16,2024-03-20,fr,[99],False,2024,a family
55375,1424442,A Family,Bir Aile,1.8657,0.000,0,2024-09-26,tr,[],False,2024,a family
4798,764619,Abigail,Abigail,1.7644,1.000,1,2019-01-07,en,[27],False,2019,abigail
354,575094,Abigail,Abigail,4.6671,6.160,269,2019-08-22,en,"[12, 14, 9648]",False,2019,abigail
1523,648408,Abigail,Abigail,2.4375,5.167,6,2019-11-22,en,[27],False,2019,abigail
54199,1624269,About Time,About Time,1.9898,0.000,0,2024-01-12,en,[],False,2024,about time


supprimer en gardant le film le plus voté

In [98]:
nb0 = len(df)

df = df.sort_values(["vote_count", "popularity"], ascending=False)
to_drop = valid.reindex(df.index) & df.duplicated(subset=key, keep="first")

removed = df[to_drop]
df = df[~to_drop].sort_index().drop(columns="title_norm")

print("supprimés :", nb0 - len(df), f"({(nb0 - len(df)) / nb0 * 100:.1f}%) | restants :", len(df))
removed[["tmdb_id", "title", "release_year", "original_language", "vote_count"]].head(20)

supprimés : 2553 (3.2%) | restants : 76714


,tmdb_id,title,release_year,original_language,vote_count
70100,1698863,The Odyssey,2026,en,595
30588,759054,Rise,2022,en,315
1881,547539,Sweetheart,2019,fr,261
10339,661950,Alone,2020,en,213
10735,746817,The Call,2020,en,175
40496,949229,Leo,2023,ta,148
272,569485,Skin,2019,en,146
4109,594082,Mosul,2019,en,134
59355,1001089,Brothers,2024,fr,117
30580,960139,The House,2022,fr,114


In [99]:
tmp = df.assign(title_norm=df["title"].map(normalize_title))
v = tmp["title_norm"] != ""
print("doublons restants :", (v & tmp.duplicated(subset=["title_norm", "release_year"])).sum())   # doit afficher 0
print("doublons tmdb_id restants :", df.duplicated(subset="tmdb_id").sum())                        # doit afficher 0

doublons restants : 0
doublons tmdb_id restants : 0


genre_ids is stored as text after reading the csv (e.g. "[28, 12]") instead of a real list
we convert it back into a python list using ast.literal_eval

In [100]:
df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
df["genre_ids"].head(5)

0    [12, 878, 28]
1     [35, 53, 18]
2     [35, 18, 53]
3     [80, 53, 18]
4    [28, 12, 878]
Name: genre_ids, dtype: object

additional check: vote_average should always be between 0 and 10 on TMDB
if not, that would indicate corrupted data from the API

In [101]:
print("Min vote_average:", df["vote_average"].min())
print("Max vote_average:", df["vote_average"].max())
print("Any negative popularity?", (df["popularity"] < 0).any())
print("Any negative vote_count?", (df["vote_count"] < 0).any())

Min vote_average: 0.0
Max vote_average: 10.0
Any negative popularity? False
Any negative vote_count? False


checking which languages are present, to spot any anomaly (unexpected codes)

In [102]:
df["original_language"].value_counts()

original_language
en    37578
es     6922
fr     5406
pt     3677
de     2527
      ...  
ti        1
io        1
rm        1
lg        1
tt        1
Name: count, Length: 144, dtype: int64

final verification of data types before saving

In [103]:
df["tmdb_id"] = df["tmdb_id"].astype(int)
df["adult"] = df["adult"].astype(bool)
df.dtypes

tmdb_id                       int64
title                           str
original_title                  str
popularity                  float64
vote_average                float64
vote_count                    int64
release_date         datetime64[us]
original_language               str
genre_ids                    object
adult                          bool
release_year                  int32
dtype: object

In [104]:
# la colonne adult est quasi constante (False) et sans équivalent dans RT/Numbers
df = df.drop(columns=["adult"])
df.columns.tolist()

['tmdb_id',
 'title',
 'original_title',
 'popularity',
 'vote_average',
 'vote_count',
 'release_date',
 'original_language',
 'genre_ids',
 'release_year']

In [105]:
# reordering columns for better readability
df = df[[
    "tmdb_id", "title", "original_title", "release_date", "release_year",
    "genre_ids", "popularity", "vote_average", "vote_count",
    "original_language"
]]
df.head(5)

,tmdb_id,title,original_title,release_date,release_year,genre_ids,popularity,vote_average,vote_count,original_language
0,299534,Avengers: Endgame,Avengers: Endgame,2019-04-24,2019,"[12, 878, 28]",98.7212,8.200,28768,en
1,496243,Parasite,기생충,2019-05-30,2019,"[35, 53, 18]",35.6564,8.490,21362,ko
2,466272,Once Upon a Time... in Hollywood,Once Upon a Time... in Hollywood,2019-07-24,2019,"[35, 18, 53]",34.9897,7.419,15284,en
3,475557,Joker,Joker,2019-10-01,2019,"[80, 53, 18]",34.4579,8.100,28367,en
4,429617,Spider-Man: Far From Home,Spider-Man: Far From Home,2019-06-28,2019,"[28, 12, 878]",33.8307,7.392,17646,en


In [106]:
import os
os.makedirs("../../data/processed", exist_ok=True)

print(df.shape)
df.to_csv("../../data/processed/tmdb_clean.csv", index=False)
print("Saved to data/processed/tmdb_clean.csv")

(76714, 10)


Saved to data/processed/tmdb_clean.csv
